In [14]:
!pip install transformers datasets scikit-learn -q

In [15]:
from google.colab import files
import json

uploaded = files.upload()  # train.json + test.json upload করো

with open("train.json") as f:
    train_data = json.load(f)
with open("test.json") as f:
    test_data = json.load(f)

Saving test.json to test (1).json
Saving train.json to train (1).json


In [16]:
def format_input(example):
    # পুরো dialogue দাও, শুধু current utterance বাদে
    history = example['dialogue'][:-1]
    context_text = " [SEP] ".join(
        [f"{t['speaker']}: {t['text']}" for t in history]
    )
    target = example['current_text']
    return f"{context_text} [SEP] TARGET: {target}"

In [17]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

tokenizer = AutoTokenizer.from_pretrained("mental/mental-roberta-base")

train_split, val_split = train_test_split(
    train_data, test_size=0.1, random_state=42,
    stratify=[d['label'] for d in train_data]
)
print(f"Train: {len(train_split)}, Val: {len(val_split)}")

class DefenseDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512, is_test=False):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = format_input(self.data[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        result = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }
        if not self.is_test:
            result['labels'] = torch.tensor(
                self.data[idx]['label'], dtype=torch.long
            )
        return result

train_dataset = DefenseDataset(train_split, tokenizer)
val_dataset   = DefenseDataset(val_split, tokenizer)
test_dataset  = DefenseDataset(test_data, tokenizer, is_test=True)

Train: 1677, Val: 187


In [18]:
import numpy as np
from transformers import AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(
    "mental/mental-roberta-base",
    num_labels=9
)

label_list = [d['label'] for d in train_split]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(9),
    y=label_list
)
weights = torch.tensor(class_weights, dtype=torch.float).to('cuda')
print("Class weights:", np.round(class_weights, 2))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mental/mental-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: [0.7  1.92 3.39 2.09 2.45 4.33 1.2  0.21 7.45]


In [20]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro':    f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'accuracy':    float((preds == labels).mean())
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='./mentalroberta-psydef',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    lr_scheduler_type='cosine',
    logging_steps=50,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [21]:
trainer.train()

metrics = trainer.evaluate(val_dataset)
print("\n=== VAL RESULTS ===")
print(f"F1 Macro:    {metrics['eval_f1_macro']:.4f}")
print(f"F1 Weighted: {metrics['eval_f1_weighted']:.4f}")
print(f"Accuracy:    {metrics['eval_accuracy']:.4f}")

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted,Accuracy
1,2.153503,2.147922,0.104620,0.373251,0.475936
2,1.981049,2.011197,0.161248,0.442708,0.459893
3,1.880645,1.968903,0.227504,0.516225,0.502674
4,1.622007,1.937675,0.238357,0.517228,0.502674
5,1.532072,1.927115,0.244729,0.494281,0.470588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


=== VAL RESULTS ===
F1 Macro:    0.2447
F1 Weighted: 0.4943
Accuracy:    0.4706


In [11]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro':    f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'accuracy':    (preds == labels).mean()
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='./mentalbert-psydef',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',       # fixed
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=50,
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer.train()

# train শেষে val evaluate করো
metrics = trainer.evaluate(val_dataset)
print("Val F1 Macro:", metrics['eval_f1_macro'])
print("Val Accuracy:", metrics['eval_accuracy'])

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted,Accuracy
1,4.365687,2.194395,0.103128,0.397144,0.540107
2,4.203993,2.124045,0.174186,0.419546,0.454545
3,3.874922,2.031596,0.158971,0.397677,0.438503
4,3.535856,1.892083,0.268123,0.536307,0.561497
5,3.297393,1.907744,0.235614,0.521271,0.577540
6,2.986617,1.827295,0.267522,0.508198,0.524064
7,2.831718,1.789227,0.257450,0.540196,0.561497
8,2.645389,1.821987,0.281862,0.540669,0.545455
9,2.499243,1.804047,0.277261,0.513519,0.524064
10,2.417124,1.806103,0.299036,0.533755,0.540107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Val F1 Macro: 0.2990360450527023
Val Accuracy: 0.5401069518716578


In [13]:
import zipfile

# predict
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)


for i, entry in enumerate(test_data):
    entry['label'] = int(preds[i])

# save
with open('prediction.json', 'w') as f:
    json.dump(test_data, f)

# zip
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('prediction.json')

print("Done. submission.zip ready.")
files.download('submission.zip')

Done. submission.zip ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# val set evaluate করো
metrics = trainer.evaluate(val_dataset)
print(metrics)

NameError: name 'trainer' is not defined